# 023 — Weak D from 022 + frozen H/A from 020

D: frozen 022 HistGradientBoosting trained on expanded broadcast weak labels outside the public five minutes plus 20-second guards. Identical threshold .5, LIVE gap <=10 frames, D duration >=25 frames. No 016 predictions, weights, or public-supervised parents. / D: неизменный бустинг 022 на слабых broadcast-метках вне публичного окна с отступами; порог и фильтр неизменны, 016 не используется.

H/A: frozen confident/contested rules from 007/020 with cleaned tracking and direct corridor assignment only when the chosen team's line is beyond 16.5 m from its own goal. Same-team anchors bridge without a turnover; missing labels filled from nearest-time evidence within live intervals, then same-half evidence. No X remains; outside official half spans D. / H/A: замороженные правила 007/020, чистка трекинга, коридор с поправкой на штрафную, заполнение неопределённостей без X.

IMPORTANT: this is NOT an entirely label-independent pipeline. H/A parameters and strategy were previously selected using the public prefix. They are merely held fixed here. D does not directly use public labels, but all experiments have historical research adaptation. Generalization cannot be established on the repeatedly inspected five minutes. / ВАЖНО: H/A ранее выбирались по публичным меткам. Здесь нового подбора нет, но всю цепочку нельзя называть независимой от разметки. Обобщение на полном матче неизвестно.

Run All with local 008 feature cache, trusted sol/022_broadcast_expanded_weak/model.pkl, ATD assets and 007 notebook/config. This notebook performs inference only; it does not train models or search settings. Public labels are loaded only after the final predictions have been saved and hashed. Full offline context. Outputs and executed notebook remain in local_review/experiments/023_weak022_anchors020. / Run All с локальными данными и артефактами; только инференс, публичные метки читаются после сохранения и хеширования прогнозов. Полный офлайн-контекст.

Only clean code/configuration is published; restricted data, predictions, metrics and videos remain local. D weights are referenced from 022, not duplicated. / Публикуем только код и конфигурацию, веса D берём из 022; результаты локальные.


In [ ]:
from pathlib import Path
import json,sys,hashlib,pickle,time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
sys.path.insert(0,str(ROOT))
from games import GAME,GAMES,half_video_blocks,asset_paths
from evaluation import possession_metrics as pm
EXPERIMENT='023_weak022_anchors020'
OUT=ROOT/'local_review/experiments'/EXPERIMENT;OUT.mkdir(parents=True,exist_ok=True)
N=199319;FPS=25;BLOCKS=half_video_blocks(GAME)
def progress(stage,**kw):(OUT/'progress.json').write_text(json.dumps(dict(stage=stage,**kw),indent=2))
# Recompute D from trusted frozen 022 weights; never load 016 or public labels.
# Считаем D из неизменных весов 022; 016 и публичные метки не загружаем.
progress('022 inference')
parent_dir=ROOT/'sol/022_broadcast_expanded_weak'
d_config=json.loads((parent_dir/'model_config.json').read_text())
assert d_config['public_labels_used_for_training'] is False
assert d_config['public_labels_used_for_selection'] is False
feature_cache=ROOT/'local_review/experiments/008_weak_dead_boosting'
feature_names=json.loads((feature_cache/'feature_names.json').read_text())
assert feature_names==d_config['feature_names']
X=np.load(feature_cache/'features.npy',mmap_mode='r')
assert X.shape==(N,len(feature_names))
with (parent_dir/'model.pkl').open('rb') as f:dmodel=pickle.load(f)
probability=np.empty(N,np.float32)
for a in tqdm(range(0,N,10000),desc='Frozen 022 / Неизменная 022'):
 b=min(N,a+10000);probability[a:b]=dmodel.predict_proba(X[a:b])[:,1]
assert np.isfinite(probability).all()
def duration_filter(raw):
 joined=raw.copy()
 edges=np.r_[0,np.flatnonzero(raw[1:]!=raw[:-1])+1,len(raw)]
 for a,b in zip(edges[:-1],edges[1:]):
  if not raw[a] and a>0 and b<len(raw) and b-a<=10:joined[a:b]=True
 result=joined.copy()
 edges=np.r_[0,np.flatnonzero(joined[1:]!=joined[:-1])+1,len(joined)]
 for a,b in zip(edges[:-1],edges[1:]):
  if joined[a] and b-a<25:result[a:b]=False
 return result
filtered=np.zeros(N,bool)
for _,a,b in BLOCKS:filtered[a:b+1]=duration_filter(probability[a:b+1]>=.5)
reference_dir=ROOT/'local_review/experiments/022_broadcast_expanded_weak'
if (reference_dir/'p_D.npy').exists():
 assert np.array_equal(probability,np.load(reference_dir/'p_D.npy'))
 assert np.array_equal(filtered,np.load(reference_dir/'dead_mask.npy'))
np.save(OUT/'p_D.npy',probability);np.save(OUT/'dead_mask.npy',filtered)
dead=filtered.copy()
del X,dmodel


In [ ]:
# Run preparation and algorithm definitions only, keeping all outputs in 023.
# Запускаем только подготовку и определения 007; все выходы остаются в 023.
progress('007 preparation')
parent007=json.loads((ROOT/'sol/007_clean_anchors/experiment.ipynb').read_text())
for index in [1,3,6,8]:
 source=''.join(parent007['cells'][index]['source'])
 if index==1:
  source=source.replace('EXPERIMENT = "007_clean_anchors"','EXPERIMENT = "023_weak022_anchors020"')
  source=source.replace('ROOT / "sol" / EXPERIMENT / "grid.json"','ROOT / "sol/007_clean_anchors/grid.json"')
  source=source.replace('import matplotlib.pyplot as plt\n','')
 if index==3:
  # Legacy event D is unused: omit event loading entirely.
  # Событийный D не используется: события не читаем.
  source=source.split('# A set-piece interval provides context only;')[0]
 exec(compile(source,f'007 cell {index}','exec'))
# Gate the direct corridor assignment by the CHOSEN team's own defensive line.
# Прямое правило коридора разрешено только по линии ВЫБРАННОЙ команды.
# The line must be strictly beyond its own penalty-area depth, toward midfield.
# Линия должна быть строго дальше границы своей штрафной в сторону центра поля.
PENALTY_DEPTH_M=16.5
BOUNDARY_TOLERANCE_M=1e-9
def line_is_advanced(line_x,defends_left,length):
 depth=np.asarray(line_x)*length if defends_left else (1-np.asarray(line_x))*length
 return np.isfinite(depth)&(depth>PENALTY_DEPTH_M+BOUNDARY_TOLERANCE_M)
# Mirror-invariant checks: inside, exactly on boundary, beyond, and missing.
# Проверки симметрии сторон: внутри, на границе, снаружи и пропуск.
probe=np.array([10.,16.5,16.51,30.,np.nan])/LENGTH
for left in [True,False]:
 test=probe if left else 1-probe
 assert np.array_equal(line_is_advanced(test,left,LENGTH),[False,False,True,True,False])
original_outside_owner=outside_owner.copy()
line_eligible=np.zeros((N,2),bool)
for half,first,last in BLOCKS:
 for team in [0,1]:
  line_eligible[first:last+1,team]=line_is_advanced(last_outfield[first:last+1,team],sides[half][team]=='left',LENGTH)
had_direct=outside_owner!=3
rows=np.flatnonzero(had_direct)
allowed=np.zeros(N,bool);allowed[rows]=line_eligible[rows,outside_owner[rows]]
# Removing forced evidence returns the frame to ordinary confident/contested processing.
# Убираем принудительное свидетельство: кадр возвращается к обычным уверенным/спорным.
outside_owner[had_direct&~allowed]=3
assert np.all(outside_owner[allowed]==original_outside_owner[allowed])
assert np.all(outside_owner[had_direct&~allowed]==3)
np.save(OUT/'corridor_eligible.npy',allowed)
np.save(OUT/'corridor_suppressed.npy',had_direct&~allowed)

# Replace legacy D completely before running the H/A solver.
# Полностью заменяем старый D перед запуском алгоритма H/A.
dead=filtered.copy()
selected=json.loads((ROOT/'sol/007_clean_anchors/selected_training_config.json').read_text())
progress('023 full inference')
labels,stats,gaps=predict_interval(selected,0,N,collect=True)
original_x=labels=='X';fallback_counts={};graded=np.zeros(N,bool)
def fill_nearest_time(values):
 valid=np.flatnonzero(np.isin(values,['H','A']))
 assert len(valid)>0
 missing=np.flatnonzero(~np.isin(values,['H','A']))
 if len(missing):
  right=np.searchsorted(valid,missing);lo=valid[np.maximum(right-1,0)];hi=valid[np.minimum(right,len(valid)-1)]
  chosen=np.where(missing-lo<=hi-missing,lo,hi)
  values[missing]=values[chosen]
 return values
# Fill only missing labels. Never change a confident anchor or a D decision.
# Заполняем только отсутствующие метки: уверенные якоря и решения D неизменны.
for half,first,last in BLOCKS:
 graded[first:last+1]=True
 nearest_evidence=np.full(last-first+1,'X',dtype='<U1')
 d=nearest[first:last+1];available=np.isfinite(d).any(axis=1)
 winner=np.argmin(d,axis=1)
 nearest_evidence[available]=np.array(['H','A'])[winner[available]]
 forced=outside_owner[first:last+1]!=3
 nearest_evidence[forced]=np.array(['H','A'])[outside_owner[first:last+1][forced]]
 assert np.isin(nearest_evidence,['H','A']).any(), 'No team evidence in entire half'
 half_prior=fill_nearest_time(nearest_evidence.copy())
 counts=dict(anchor_edge_fill=0,nearest_ball_fallback=0,same_half_fallback=0)
 states=dead[first:last+1]
 for a,b in runs_of(states):
  if states[a]:continue
  lo,hi=first+a,first+b;part=labels[lo:hi].copy();missing=part=='X'
  if not missing.any():continue
  if np.isin(part,['H','A']).any():
   counts['anchor_edge_fill']+=int(missing.sum());part=fill_nearest_time(part)
  else:
   evidence=nearest_evidence[a:b].copy()
   if np.isin(evidence,['H','A']).any():
    counts['nearest_ball_fallback']+=len(part);part=fill_nearest_time(evidence)
   else:
    counts['same_half_fallback']+=len(part);part=half_prior[a:b].copy()
  labels[lo:hi]=part
 fallback_counts[str(half)]=counts
labels[~graded]='D'
assert not np.any(labels=='X')
assert np.array_equal(labels[graded]=='D',filtered[graded])
assert np.array_equal(labels[graded&~original_x],predict_interval(selected,0,N)[0][graded&~original_x])
result=pd.DataFrame({'frame':np.arange(N),'label':labels});assert pm.validate_submission(result,N) is None
result.to_csv(OUT/'submission.csv',index=False)
# Exact start-of-half audit, including the second half pre-kickoff coverage.
# Точная проверка начал таймов, включая участок до второго розыгрыша.
audit={str(h):dict(first_frame=a,last_frame=b,first_label=str(labels[a]),X_frames=int((labels[a:b+1]=='X').sum()),initial_10s_X=int((labels[a:a+250]=='X').sum())) for h,a,b in BLOCKS}
(OUT/'coverage.json').write_text(json.dumps(dict(total_X=int((labels=='X').sum()),original_X_in_halves=int((original_x&graded).sum()),fallback_counts=fallback_counts,halves=audit,outside_half_policy='D'),indent=2))
print(json.dumps(audit,indent=2))

# Freeze the complete output before opening permitted reference labels.
# Фиксируем полный прогноз до чтения разрешённого эталона.
(OUT/'frozen_before_evaluation.json').write_text(json.dumps(dict(
 prediction_sha256=hashlib.sha256((OUT/'submission.csv').read_bytes()).hexdigest(),
 D_model_sha256=hashlib.sha256((ROOT/'sol/022_broadcast_expanded_weak/model.pkl').read_bytes()).hexdigest(),
 D_public_labels_used=False,HA_historically_public_selected=True,new_parameter_selection=False),indent=2))


In [ ]:
gt=pm.read_label_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')
# Metrics use only public labels, with full offline inference context.
# Метрики считаются только по публичным меткам с полным офлайн-контекстом.
def score_frames(frames):
 y=pm.label_codes(gt.set_index('frame').loc[frames,'label'].to_numpy())
 p=pm.label_codes(result.set_index('frame').loc[frames,'label'].to_numpy())
 runs=pm.valid_runs(y);cm=pm.scored_confusion(pm.confusion(y,p))
 out=dict(spellF1=pm.spell_scores(y,pm.fill_abstentions(p,runs),runs)['mean_f1'],macroF1=pm.macro_f1(cm),classes={})
 for i,label in enumerate(['H','A','D']):
  precision,recall,f1=pm.per_class_prf(cm,i)
  out['classes'][label]=dict(precision=precision,recall=recall,F1=f1,support=int(cm[i].sum()),predicted=int(cm[:,i].sum()))
 assert np.isclose(np.mean([v['F1'] for v in out['classes'].values()]),out['macroF1'])
 return out
public=gt.loc[gt.label.isin(['H','A','D']),'frame'].to_numpy()
metrics=dict(public_5min=score_frames(public),last_minute=score_frames(public[6000:]),
 context='022 D trained on weak labels outside public window; H/A historically selected on public prefix; full offline inputs; no new fit or selection; repeatedly viewed diagnostics')
(OUT/'metrics.json').write_text(json.dumps(metrics,indent=2))
config=dict(D_parent='022_broadcast_expanded_weak',D_model='model.pkl',D_public_supervision=False,HA_historical_public_selection=True,threshold=.5,min_dead_frames=25,max_live_gap_frames=10,
 HA_parent='007_clean_anchors',HA_config=selected,corridor_min_own_goal_depth_m=16.5,corridor_comparison='strictly_greater',corridor_boundary_tolerance_m=1e-9,no_X_policy='interval anchor extension, nearest-ball fallback, same-half temporal fallback; D outside halves',
 model_sha256=hashlib.sha256((ROOT/'sol/022_broadcast_expanded_weak/model.pkl').read_bytes()).hexdigest())
(OUT/'model_config.json').write_text(json.dumps(config,indent=2))
progress('completed',X_frames=0)
print(json.dumps(metrics,indent=2))

# Compare only for reporting, never for choosing a model or setting.
# Сравнение только для отчёта, без выбора модели или настройки.
prior=pd.read_csv(ROOT/'local_review/experiments/020_penalty_gated_corridor/submission.csv').label.to_numpy()
seq=[]
for name,values in [('023',labels[public]),('020',prior[public]),('reference',gt.set_index('frame').loc[public,'label'].to_numpy())]:
 for label in ['H','A','D','X']:
  seq.append(dict(model=name,label=label,sequences=int(sum(values[a]==label for a,b in runs_of(values))),frames=int((values==label).sum())))
pd.DataFrame(seq).to_csv(OUT/'sequence_counts_5min.csv',index=False)
assert np.array_equal(labels[graded]=='D',np.load(OUT/'dead_mask.npy')[graded])
assert not (labels=='X').any()
print(pd.DataFrame(seq).to_string(index=False))
